# Sommelier — Vietnamese UV Isolated Venv Test on Kaggle (2x T4 GPU)

Runs the sommelier podcast pipeline inside a **clean isolated `venv`** (`/kaggle/working/sommelier_env`) using **`uv`** on Kaggle with dual T4 GPUs.
This notebook dynamically patches runtime dependencies and handles Kaggle paths without modifying repository source files.


In [1]:
import os
os.environ["MPLBACKEND"] = "Agg"

## 1. Sanity check the Kaggle runtime & GPUs

In [2]:
!nvidia-smi
!python --version
!df -h /kaggle/working 2>/dev/null || df -h .
import torch
print('PyTorch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('Device count (GPUs):', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))


Tue Aug 18 09:06:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Setup Working Directory & Repository

In [3]:
import os
import shutil

BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
PROJECT_DIR = os.path.join(BASE_DIR, 'sommerlier')
ENV_DIR = os.path.join(BASE_DIR, 'sommelier_env')
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')

print(f"BASE_DIR: {BASE_DIR}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ENV_DIR: {ENV_DIR}")
print(f"AUDIO_DIR: {AUDIO_DIR}")




BASE_DIR: /kaggle/working
PROJECT_DIR: /kaggle/working/sommerlier
ENV_DIR: /kaggle/working/sommelier_env
AUDIO_DIR: /kaggle/working/vi_audio


In [4]:
# Luôn xóa repo cũ và clone lại
if os.path.exists(PROJECT_DIR):
    print(f"Removing old repository: {PROJECT_DIR}")
    shutil.rmtree(PROJECT_DIR)

os.chdir(BASE_DIR)
print("Cloning repository...")
!git clone -b solid-architecture https://github.com/foresst123/sommerlier.git

print("Clone completed.")

Cloning repository...
Cloning into 'sommerlier'...
remote: Enumerating objects: 678, done.
remote: Counting objects: 100% (678/678), done.
remote: Compressing objects: 100% (432/432), done.
remote: Total 678 (delta 446), reused 467 (delta 235), pack-reused 0 (from 0)
Receiving objects: 100% (678/678), 22.49 MiB | 22.04 MiB/s, done.
Resolving deltas: 100% (446/446), done.
Clone completed.


## 3. Install dependencies into isolated venv via UV (~10 min)

Mirrors the three-step install order (torch CUDA 12.6 first, then requirements).


In [ ]:
# 1. Install uv package manager
!pip install -q uv yt-dlp

# 2. Create clean isolated virtual environment
!uv venv --allow-existing {ENV_DIR}

# 3. Define proposed optimized dependencies list
PROPOSED_REQUIREMENTS = """
numpy==2.2.2
torch==2.8.0
torchaudio==2.8.0
torchvision==0.23.0
lightning==2.4.0
torchmetrics==1.6.2
onnxruntime-gpu==1.19.2
pyannote.audio==4.0.7
speechbrain==1.0.2
faster-whisper==1.2.0
whisperx==3.8.6
ctranslate2==4.5.0
demucs>=4.0.1
panns-inference
librosa==0.10.2.post1
soundfile==0.13.1
pydub==0.25.1
julius==0.2.7
numba==0.61.2
transformers==4.53.0
huggingface-hub>=0.9.8
nltk==3.9.1
openai==1.63.0
pandas==2.2.3
PyYAML==6.0.2
tqdm==4.67.1
requests==2.32.4
einops==0.8.1
hydra-core==1.3.2
omegaconf==2.3.0
setuptools>=75.0.0
accelerate==0.30.0
"""

req_file = os.path.join(BASE_DIR, 'requirements_proposed.txt')
with open(req_file, 'w') as f:
    f.write(PROPOSED_REQUIREMENTS.strip())

print('>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...')
!uv pip install --python {ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

print('\n>>> Step 2: Installing proposed dependencies into venv via UV...')
!uv pip install --python {ENV_DIR} -r {req_file} --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match

print('\n>>> Step 3: Installing DialogueSidon for Target Speaker Extraction...')
!uv pip install --python {ENV_DIR} "git+https://github.com/sarulab-speech/Sidon.git@c8cde2b24e4c77c599ad43a9871140cdc9beeffa"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 3.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 67.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.7 MB/s eta 0:00:00:00:01
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: sommelier_env
Activate with: source sommelier_env/bin/activate
>>> Step 1: Pre-installing PyTorch CUDA 12.6 inside isolated venv...
Using Python 3.12.13 environment at: sommelier_env
Resolved 29 packages in 614ms                                        
Prepared 29 packages in 55.27s                                           
░░░░░░░░░░░░░░░░░░░░ [0/29] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional,

In [6]:
# --- QWEN3-ASR ISOLATED ENVIRONMENT SETUP ---
# We create a separate environment for Qwen3-ASR to avoid huggingface-hub conflicts with WhisperX.
QWEN3_ENV_DIR = os.path.join(BASE_DIR, "qwen3_env")

!uv venv --allow-existing {QWEN3_ENV_DIR} --python 3.12

# Install PyTorch CUDA 12.6
!uv pip install --python {QWEN3_ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 --extra-index-url https://download.pytorch.org/whl/cu126

# Install Qwen3 specific dependencies (Transformers 5.13 requires newer huggingface-hub)
!uv pip install --python {QWEN3_ENV_DIR} "transformers>=5.13.0" "huggingface-hub>=1.5.0" accelerate soundfile librosa

# Verify installation
!{QWEN3_ENV_DIR}/bin/python -c "from transformers import AutoProcessor, AutoModelForMultimodalLM; print(\"✅ Qwen3 environment OK\")"


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: qwen3_env
Activate with: source qwen3_env/bin/activate
Using Python 3.12.13 environment at: qwen3_env
Resolved 29 packages in 46ms                                         
░░░░░░░░░░░░░░░░░░░░ [0/29] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 29 packages in 27.54s                             
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.6.4.1
 + nvidia-cuda-cupti-cu12==12.6.80
 + nvidia-cuda-nvrtc-cu12==12.6.77
 + nvidia-cuda-runtime-cu12==12.6.77
 + n

In [7]:
import os

BASE_DIR = "/kaggle/working"
DIARIZEN_ENV_DIR = os.path.join(BASE_DIR, "diarizen_env")
PYTHON = f"{DIARIZEN_ENV_DIR}/bin/python"

%env UV_LINK_MODE=copy

!rm -rf {DIARIZEN_ENV_DIR}
!uv venv {DIARIZEN_ENV_DIR} --python 3.12

# 1. Cài đặt PyTorch chuẩn
!uv pip install --python {PYTHON} torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# 2. Cài đặt các gói core cố định version để tránh conflict
!uv pip install --python {PYTHON} numpy==1.26.4 scipy==1.13.1 pandas==2.2.3 numba==0.59.1 llvmlite==0.42.0

# 3. Cài đặt thư viện xử lý âm thanh
!uv pip install --python {PYTHON} soundfile librosa==0.10.2.post1 matplotlib==3.9.4 pyparsing accelerate==1.12.0

# 4. TẢI VÀ CÀI ĐẶT BẢN "ĐỘ CHẾ" CỦA PYANNOTE TỪ GITHUB DIARIZEN (Cực kỳ quan trọng)
!uv pip install --python {PYTHON} "git+https://github.com/BUTSpeechFIT/DiariZen.git@844f5555b0a98acd0931511fc641a8c5b8ba92c7#subdirectory=pyannote-audio"

# 5. Cài đặt DiariZen gốc
!uv pip install --python {PYTHON} git+https://github.com/BUTSpeechFIT/DiariZen.git@844f5555b0a98acd0931511fc641a8c5b8ba92c7

# 6. Cài đặt các Dependency còn lại
!uv pip install --python {PYTHON} speechbrain==1.0.2 toml==0.10.2 wrapt==2.3.0 psutil==7.0.0

# 7. Ép lại version một lần nữa (đề phòng Pyannote tự ý nâng cấp numpy/scipy làm hỏng Numba)
!uv pip install --python {PYTHON} numpy==1.26.4 scipy==1.13.1 --no-deps

# ================= KIỂM TRA MÔI TRƯỜNG =================
!{PYTHON} -c "import sys,numpy,scipy,torch; print('Python:',sys.version.split()[0]); print('NumPy:',numpy.__version__); print('SciPy:',scipy.__version__); print('Torch:',torch.__version__); print('CUDA:',torch.version.cuda); print('GPU:',torch.cuda.is_available())"

!{PYTHON} -c "import pyannote.audio; print('Pyannote:',pyannote.audio.__version__)"

# Dòng test này sẽ in ra tham số 'config' nếu cài đặt bản độ chế thành công
!{PYTHON} -c "from pyannote.audio.pipelines import SpeakerDiarization; import inspect; print(inspect.signature(SpeakerDiarization.__init__))"

# Cuối cùng là test thử luồng khởi tạo DiariZen xem có bị báo lỗi API Mismatch nữa không
!{PYTHON} -c "from diarizen.pipelines.inference import DiariZenPipeline; print('✅ DiariZenPipeline OK')"

print("✅ Environment completed")


env: UV_LINK_MODE=copy
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: diarizen_env
Activate with: source diarizen_env/bin/activate
Using Python 3.12.13 environment at: diarizen_env
Resolved 27 packages in 727ms                                        
Prepared 27 packages in 50.73s                                           
Installed 27 packages in 6.47s                              
 + filelock==3.32.3
 + fsspec==2026.7.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.5.2
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==9.1.0.70
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-curand-cu12==10.3.2.106
 + nvidia-cusolver-cu12==11.4.5.107
 + nvidia-cusparse-cu12==12.1.0.106
 + nvidia-nccl-cu12==2.21.5
 + nvidia-nvjitlink-cu12==12.9.86
 + nvidia-nvtx-cu12==12.1.105
 + pillow==12.3.0
 + setuptools==78.1.0


## 4. Hugging Face authentication & Config update

In [8]:
# Authenticate HuggingFace Token (supports Kaggle Secrets & Interactive Input)
import json, pathlib

hf_token = ""
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Fetched HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print("Kaggle Secrets not available or HF_TOKEN secret missing.")

if not hf_token:
    from getpass import getpass
    hf_token = getpass('Enter HF Token (hf_...): ')

from huggingface_hub import login
login(token=hf_token)

# Update config.json in podcast-pipeline
cfg_path = os.path.join(PROJECT_DIR, 'podcast-pipeline', 'config.json')
if os.path.exists(cfg_path):
    cfg = json.loads(pathlib.Path(cfg_path).read_text())
    cfg['huggingface_token'] = hf_token
    pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=2, ensure_ascii=False))
    print(f'Updated Hugging Face token in {cfg_path}')


Fetched HF_TOKEN from Kaggle Secrets.
Updated Hugging Face token in /kaggle/working/sommerlier/podcast-pipeline/config.json


## 5. Prepare Audio Input (Auto-detects Kaggle Dataset `/kaggle/input`, Drag-Drop, or Fallback)

In [9]:
import os, glob, random, shutil, pathlib
import torch, torchaudio

# Always prepare a writable working audio folder
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()
AUDIO_DIR = os.path.join(BASE_DIR, 'vi_audio')
os.makedirs(AUDIO_DIR, exist_ok=True)

# 1. Auto-detect if user attached audio via Kaggle Datasets (/kaggle/input/...)
input_audio_files = []
if os.path.exists('/kaggle/input'):
    extensions = ('*.mp3', '*.wav', '*.flac', '*.m4a', '*.aac', '*.ogg')
    for ext in extensions:
        input_audio_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if input_audio_files:
    print(f"✅ Detected {len(input_audio_files)} audio file(s) in /kaggle/input/:")
    for f in input_audio_files:
        print("  -", f)
        dst = os.path.join(AUDIO_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy(f, dst)
    print(f"Sync completed into writable folder: {AUDIO_DIR}")

# 2. Scan audio files in /kaggle/working/vi_audio/
audio_extensions = ('.mp3', '.wav', '.flac', '.m4a', '.aac', '.ogg')
audio_files = [f for f in glob.glob(os.path.join(AUDIO_DIR, '*')) if f.lower().endswith(audio_extensions)]

# 3. Fallback: If no audio files exist, generate a 10s synthetic test WAV file
if not audio_files:
    print('\n⚠️ No audio files found in /kaggle/input or /kaggle/working/vi_audio/.')
    print('Generating synthetic benchmark WAV file in vi_audio for smoke testing...')
    sample_rate = 16000
    duration_sec = 10
    t = torch.linspace(0, duration_sec, sample_rate * duration_sec)
    waveform = 0.3 * torch.sin(2 * 3.14159 * 440 * t) + 0.2 * torch.sin(2 * 3.14159 * 880 * t)
    waveform = waveform.unsqueeze(0)
    
    fallback_wav = os.path.join(AUDIO_DIR, 'kaggle_test_sample.wav')
    torchaudio.save(fallback_wav, waveform, sample_rate)
    audio_files = [fallback_wav]
    print(f'✅ Fallback test WAV created at: {fallback_wav}')

print(f'\n✅ Total audio file(s) ready for pipeline: {len(audio_files)}')
for af in audio_files:
    print('  -', af)


✅ Detected 1 audio file(s) in /kaggle/input/:
  - /kaggle/input/data-thu-that-thach/thu_that_thach_10m.mp3
Sync completed into writable folder: /kaggle/working/vi_audio

✅ Total audio file(s) ready for pipeline: 1
  - /kaggle/working/vi_audio/thu_that_thach_10m.mp3


## 6. Dynamic In-Notebook Patches & Run Pipeline on 2x T4 GPUs

In [10]:
# Dynamically apply in-notebook patches without altering original repo files outside this execution
import glob, os

# Patch 1: Dynamically patch pkg_resources in venv site-packages regardless of Python version
site_pkgs = glob.glob(os.path.join(ENV_DIR, 'lib', 'python*', 'site-packages'))
if site_pkgs:
    pkg_res_file = os.path.join(site_pkgs[0], 'pkg_resources.py')
    with open(pkg_res_file, 'w', encoding='utf-8') as f:
        f.write('def declare_namespace(name): pass\n')
    print(f'✅ Dynamic patch applied to {pkg_res_file}')


✅ Dynamic patch applied to /kaggle/working/sommelier_env/lib/python3.12/site-packages/pkg_resources.py


In [25]:
# Run the pipeline configured for Kaggle 2x T4 GPU
import os
import sys

os.chdir(os.path.join(PROJECT_DIR, 'podcast-pipeline'))
python_bin = os.path.join(ENV_DIR, 'bin', 'python')

site_packages = os.path.join(ENV_DIR, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
nvidia_lib = f"{site_packages}/nvidia/cudnn/lib:{site_packages}/torch/lib"

os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + nvidia_lib

# Sử dụng $$f thay vì $f để chạy đúng cú pháp Bash vòng lặp trong Jupyter
!for f in {AUDIO_DIR}/*; do \
    if [ -f "$$f" ]; then \
        CUDA_VISIBLE_DEVICES=0,1 {python_bin} main.py \
        --audio "$$f" \
        --env kaggle \
        --lang vi \
        --tse \
        --panns \
        --vad \
        --llm_refinement \
        --ASRMoE; \
    fi; \
done




Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
/kaggle/working/sommelier_env/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:54: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()
INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [disable_jit_profiling, allow_tf32]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []
--2026-08-18 09:18:00--  http://storage.googleapis.com/us_audioset/youtube_corpus/v1/c

## 7. Inspect Output

In [28]:
import glob, json, pathlib

# Dùng thẳng đường dẫn tuyệt đối không thông qua biến nào cả
result_folder = '/kaggle/working/vi_audio/_final'
json_paths = sorted(glob.glob(f'{result_folder}/**/*.json', recursive=True))

print(f'Found {len(json_paths)} result file(s):')
for p in json_paths:
    print(' -', p)

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    print('\nMetadata:')
    print(json.dumps(result.get('metadata', {}), indent=2, ensure_ascii=False))


Found 0 result file(s):


## Kaggle 2x T4 Troubleshooting & Tips

- **Dual GPU Allocation**: Specified `CUDA_VISIBLE_DEVICES=0,1` for multi-GPU runtime.
- **Kaggle Secrets**: Set `HF_TOKEN` in Kaggle Secrets (Add-ons -> Secrets) so Hugging Face models auto-authenticate.
- **Input Audio**: Supports Kaggle Datasets (`/kaggle/input/...`), direct drag-drop (`/kaggle/working/vi_audio/`), or synthetic benchmark fallback audio.
- **Isolated Venv**: `uv` isolates dependencies in `/kaggle/working/sommelier_env` to avoid Kaggle pre-installed package conflicts.


In [35]:
import pandas as pd

if json_paths:
    result = json.loads(pathlib.Path(json_paths[0]).read_text())
    audio_name = result.get("metadata", {}).get("audio_name", pathlib.Path(json_paths[0]).stem)
    
    rows = []
    for seg in result.get("segments", []):
        # Format time
        start = seg.get("start", 0)
        end = seg.get("end", 0)
        time_str = f"{start:.2f} - {end:.2f}"
        
        # Get speaker
        speaker = seg.get("speaker", "Unknown")
        
        # Get text (fallback to whisper or ensemble if text is not available)
        text = seg.get("text") or seg.get("text_ensemble") or seg.get("text_whisper") or ""
        
        rows.append({
            "Speaker": speaker,
            "Time": time_str,
            "Audio": audio_name,
            "Text": text
        })
    
    df = pd.DataFrame(rows)
    # Configure pandas display to show full text
    pd.set_option("display.max_colwidth", None)
    display(df)
else:
    print("No results found to display.")


No results found to display.


In [36]:
!zip -r /kaggle/working/vi_audio_output.zip /kaggle/working/vi_audio


  adding: kaggle/working/vi_audio/ (stored 0%)
  adding: kaggle/working/vi_audio/thu_that_thach_10m.mp3 (deflated 29%)
